In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, classification_report,
                              confusion_matrix)

In [ ]:
# Load data
train = pd.read_csv("../data/final/train.csv")
val   = pd.read_csv("../data/final/val.csv")
test  = pd.read_csv("../data/final/test.csv")

X_train, y_train = train['input_text'], train['cognitive_label']
X_val,   y_val   = val['input_text'],   val['cognitive_label']
X_test,  y_test  = test['input_text'],  test['cognitive_label']

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 3624 | Val: 777 | Test: 777


In [3]:
# MODEL 1 — Logistic Regression + TF-IDF
print("\nTraining Logistic Regression...")

lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,     # Top 20k most important words
        ngram_range=(1, 2),     # Use single words AND word pairs
        sublinear_tf=True,      # Apply log scaling to term frequencies
        min_df=3,               # Ignore words appearing < 3 times
        strip_accents='unicode',
        analyzer='word'
    )),
    ('clf', LogisticRegression(
        C=1.0,                  # Regularization strength
        max_iter=1000,
        class_weight='balanced',# Handle any remaining imbalance
        solver='lbfgs',
        random_state=42
    ))
])

lr_pipeline.fit(X_train, y_train)

# Evaluate on validation set
lr_val_preds  = lr_pipeline.predict(X_val)
lr_test_preds = lr_pipeline.predict(X_test)

lr_val_acc  = accuracy_score(y_val,  lr_val_preds)
lr_test_acc = accuracy_score(y_test, lr_test_preds)
lr_test_f1  = f1_score(y_test, lr_test_preds, average='weighted')

print(f"  Val  Accuracy : {lr_val_acc:.4f}")
print(f"  Test Accuracy : {lr_test_acc:.4f}")
print(f"  Test F1       : {lr_test_f1:.4f}")

# Save model
joblib.dump(lr_pipeline, "../models/baseline_logreg.pkl")
print("  Saved → ../models/baseline_logreg.pkl")


Training Logistic Regression...
  Val  Accuracy : 0.9318
  Test Accuracy : 0.9369
  Test F1       : 0.9369
  Saved → ../models/baseline_logreg.pkl


In [4]:
# MODEL 2 — SVM + TF-IDF
print("\nTraining SVM...")

svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=3
    )),
    ('clf', LinearSVC(
        C=0.5,
        max_iter=2000,
        class_weight='balanced',
        random_state=42
    ))
])

svm_pipeline.fit(X_train, y_train)

svm_val_preds  = svm_pipeline.predict(X_val)
svm_test_preds = svm_pipeline.predict(X_test)

svm_val_acc  = accuracy_score(y_val,  svm_val_preds)
svm_test_acc = accuracy_score(y_test, svm_test_preds)
svm_test_f1  = f1_score(y_test, svm_test_preds, average='weighted')

print(f"  Val  Accuracy : {svm_val_acc:.4f}")
print(f"  Test Accuracy : {svm_test_acc:.4f}")
print(f"  Test F1       : {svm_test_f1:.4f}")

joblib.dump(svm_pipeline, "../models/baseline_svm.pkl")
print("  Saved → ../models/baseline_svm.pkl")


Training SVM...
  Val  Accuracy : 0.9344
  Test Accuracy : 0.9356
  Test F1       : 0.9356
  Saved → ../models/baseline_svm.pkl


In [ ]:
# RESULTS SUMMARY

print("\n" + "="*50)
print("   BASELINE RESULTS (on TEST set)")
print("="*50)

baseline_results = {
    "Logistic Regression": {
        "accuracy": lr_test_acc, "f1": lr_test_f1,
        "precision": precision_score(y_test, lr_test_preds, average='weighted'),
        "recall": recall_score(y_test, lr_test_preds, average='weighted'),
        "preds": lr_test_preds
    },
    "SVM": {
        "accuracy": svm_test_acc, "f1": svm_test_f1,
        "precision": precision_score(y_test, svm_test_preds, average='weighted'),
        "recall": recall_score(y_test, svm_test_preds, average='weighted'),
        "preds": svm_test_preds
    }
}

for name, metrics in baseline_results.items():
    print(f"\n  {name}:")
    print(f"    Accuracy  : {metrics['accuracy']:.4f}")
    print(f"    F1        : {metrics['f1']:.4f}")
    print(f"    Precision : {metrics['precision']:.4f}")
    print(f"    Recall    : {metrics['recall']:.4f}")

print("\n  Full classification report — Logistic Regression:")
print(classification_report(y_test, lr_test_preds,
      target_names=["System 2 (rational)", "System 1 (emotional)"]))

# Save baseline results for comparison later
pd.DataFrame(baseline_results).T.drop(columns='preds').to_csv(
    "../outputs/reports/baseline_results.csv"
)
print("\nSaved → ../outputs/reports/baseline_results.csv")


   BASELINE RESULTS (on TEST set)

  Logistic Regression:
    Accuracy  : 0.9369
    F1        : 0.9369
    Precision : 0.9376
    Recall    : 0.9369

  SVM:
    Accuracy  : 0.9356
    F1        : 0.9356
    Precision : 0.9358
    Recall    : 0.9356

  Full classification report — Logistic Regression:
                      precision    recall  f1-score   support

 System 2 (rational)       0.95      0.92      0.94       389
System 1 (emotional)       0.92      0.96      0.94       388

            accuracy                           0.94       777
           macro avg       0.94      0.94      0.94       777
        weighted avg       0.94      0.94      0.94       777


Saved → ../outputs/reports/baseline_results.csv
